In [1]:
import pandas as pd
import numpy as np

In [2]:
profit_loss = pd.read_csv(
    "exports/clean_profit_loss.csv"
)

balance_sheet = pd.read_csv(
    "exports/clean_balance_sheet.csv"
)

cash_flow = pd.read_csv(
    "exports/clean_cash_flow.csv"
)

In [3]:
results = []

In [4]:
companies = profit_loss["company_id"].unique()

In [5]:
for company in companies:

    pros = []

    cons = []

In [7]:
balance_sheet["de_ratio"] = (
    balance_sheet["borrowings"] /
    (
        balance_sheet["equity_capital"] +
        balance_sheet["reserves"] +
        1
    )
)

In [8]:
    company_bs = (
        balance_sheet[
            balance_sheet["company_id"] == company
        ]
        .sort_values("year")
    )

    latest_de = (
        company_bs["de_ratio"]
        .dropna()
        .iloc[-1]
        if len(company_bs) > 0
        else np.nan
    )

    if latest_de < 0.1:

        pros.append(
            "Company is almost debt free."
        )

In [10]:
# =========================
# Derived Financial Metrics
# =========================

# ROE
balance_sheet["roe"] = (
    balance_sheet["reserves"] /
    (balance_sheet["equity_capital"] + 1)
) * 100

# Debt to Equity
balance_sheet["de_ratio"] = (
    balance_sheet["borrowings"] /
    (
        balance_sheet["equity_capital"] +
        balance_sheet["reserves"] +
        1
    )
)

# Net Margin
profit_loss["net_margin"] = (
    profit_loss["net_profit"] /
    (profit_loss["sales"] + 1)
) * 100

# Operating Profit Margin
profit_loss["opm_percentage"] = (
    profit_loss["operating_profit"] /
    (profit_loss["sales"] + 1)
) * 100

# Interest Coverage
profit_loss["interest_coverage"] = (
    profit_loss["operating_profit"] /
    (profit_loss["interest"] + 1)
)

# Cash Conversion
cash_merged = cash_flow.merge(
    profit_loss[
        [
            "company_id",
            "year",
            "net_profit"
        ]
    ],
    on=["company_id", "year"],
    how="left"
)

cash_merged["cash_conversion"] = (
    cash_merged["operating_activity"] /
    (cash_merged["net_profit"] + 1)
)

In [14]:
company_bs = (
    balance_sheet[
        balance_sheet["company_id"] == company
    ]
    .sort_values("year")
)

In [15]:
avg_roe = (
    company_bs["roe"]
    .tail(3)
    .mean()
)

if avg_roe > 20:
    pros.append(
        f"Company has a good return on equity (ROE) track record: 3 Years ROE {avg_roe:.2f}%"
    )

In [16]:
    company_pl = (
        profit_loss[
            profit_loss["company_id"] == company
        ]
        .sort_values("year")
    )

    dividend_avg = (
        company_pl["dividend_payout"]
        .tail(5)
        .mean()
    )

    if dividend_avg > 30:

        pros.append(
            f"Company has been maintaining a healthy dividend payout of {dividend_avg:.2f}%"
        )

In [17]:
    sales = (
        company_pl["sales"]
        .dropna()
    )

In [18]:
    if len(sales) >= 10:

        sales_cagr = (

            (
                sales.iloc[-1] /
                (sales.iloc[-10] + 1)
            ) ** (1/10) - 1

        ) * 100

        if sales_cagr > 15:

            pros.append(
                f"Strong long-term revenue growth of {sales_cagr:.2f}% CAGR over 10 years"
            )

In [19]:
    opm = (
        company_pl["opm_percentage"]
        .tail(3)
        .values
    )

    if len(opm) == 3:

        if opm[0] < opm[1] < opm[2]:

            pros.append(
                "Improving operating margins consistently for 3 years"
            )

In [20]:
    company_cf = (
        cash_flow[
            cash_flow["company_id"] == company
        ]
        .sort_values("year")
    )

In [21]:
    merged = company_cf.merge(
        company_pl[
            [
                "year",
                "net_profit"
            ]
        ],
        on="year"
    )

In [22]:
    recent = merged.tail(3)

    if all(
        recent["operating_activity"] >
        recent["net_profit"]
    ):

        pros.append(
            "Strong cash conversion — OCF exceeds reported profits"
        )

In [23]:
    profits = (
        company_pl["net_profit"]
        .dropna()
    )

In [24]:
    if len(profits) >= 3:

        profit_cagr = (

            (
                profits.iloc[-1] /
                (profits.iloc[-3] + 1)
            ) ** (1/3) - 1

        ) * 100

        if profit_cagr > 15:

            pros.append(
                f"Profit growth accelerated significantly at {profit_cagr:.2f}% CAGR"
            )

C:\Users\sujit\AppData\Local\Temp\ipykernel_15660\3568748543.py:5: RuntimeWarning: invalid value encountered in scalar power
  (


In [25]:
    if len(sales) >= 5:

        sales_cagr_5 = (

            (
                sales.iloc[-1] /
                (sales.iloc[-5] + 1)
            ) ** (1/5) - 1

        ) * 100

        if sales_cagr_5 < 10:

            cons.append(
                "Below-average sales growth over past five years"
            )

In [26]:
    borrowings = (
        company_bs["borrowings"]
        .dropna()
    )

In [27]:
    if len(borrowings) >= 2:

        if borrowings.iloc[-1] > (
            borrowings.iloc[-2] * 1.5
        ):

            cons.append(
                "Borrowings have increased significantly in the recent year"
            )

In [28]:
    if len(opm) == 3:

        if opm[0] > opm[1] > opm[2]:

            cons.append(
                "Operating margins have been declining for three consecutive years"
            )

In [29]:
    if latest_de > 1.5:

        cons.append(
            "Stock carries high debt — leverage levels require monitoring"
        )

In [30]:
    if len(recent) > 0:

        earnings_gap = (
            recent["net_profit"] -
            recent["operating_activity"]
        ).mean()

        if earnings_gap > (
            0.3 * recent["net_profit"].mean()
        ):

            cons.append(
                "Earnings quality concern — reported profits exceed actual cash generation"
            )